# TIỀN XỬ LÝ DỮ LIỆU
## Brazilian E‑Commerce Public Dataset by Olist

**Mục tiêu:**
- Phân tích khám phá dữ liệu (EDA) bộ dữ liệu thương mại điện tử Brazil
- Thực hiện tiền xử lý dữ liệu có hệ thống
- Chuẩn bị dữ liệu cho bài toán phân loại nhị phân
- Xây dựng và so sánh các mô hình học máy được giảng dạy trong đại học

---

## MỤC LỤC

1. [Import thư viện & Cấu hình](#1)
2. [Load & Chuẩn bị dữ liệu](#2)
3. [Khám phá bảng đơn hàng chính](#3)
4. [Phân tích biến phân loại chính](#4)
5. [Xử lý & Chuyển đổi dữ liệu](#5)
6. [Phân tích xu hướng theo thời gian](#6)
7. [Phân tích phân phối biến liên tục](#7)
8. [Tạo biến mục tiêu (binary)](#8)
9. [Kết hợp dữ liệu – Master Dataset](#9)
10. [Encoding & Scaling](#10)
11. [Mô hình – Naive Bayes](#11)
12. [Mô hình – K‑Nearest Neighbors (KNN)](#12)
13. [Mô hình – Decision Tree](#13)
14. [Mô hình – Support Vector Machine (SVM)](#14)
15. [So sánh kết quả các mô hình](#15)
16. [Kết luận & Insights](#16)

---

## 1. IMPORT THƯ VIỆN & CẤU HÌNH

**Mục đích:**
* Nạp toàn bộ thư viện cần thiết cho dự án
* Thiết lập cấu hình đồ họa và hiển thị nhất quán
* Thiết lập seed để tái lập kết quả

**Lưu ý:**
* Sử dụng matplotlib + seaborn cho trực quan hóa
* Scikit‑learn cho tiền xử lý và mô hình ML

In [ ]:
# ===== 1. Thư viện xử lý dữ liệu =====
import numpy as np
import pandas as pd

# ===== 2. Thư viện trực quan hóa =====
import matplotlib.pyplot as plt
import seaborn as sns

# ===== 3. Thư viện tiền xử lý =====
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ===== 4. Thư viện mô hình học máy =====
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

# ===== 5. Thư viện đánh giá =====
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ===== 6. Cài đặt cấu hình chung =====
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid", palette="husl")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Tất cả thư viện đã được nạp thành công!")
print(f"   NumPy   : {np.__version__}")
print(f"   Pandas  : {pd.__version__}")
print(f"   Seaborn : {sns.__version__}")

### NHẬN XÉT:

**Quan sát:**
- Toàn bộ thư viện được nạp thành công, bao gồm pandas, numpy, matplotlib, seaborn và scikit‑learn
- Random seed = 42 đảm bảo tính tái lập của kết quả

**Ý nghĩa:**
- Nền tảng kỹ thuật được chuẩn bị đầy đủ cho các bước xử lý và mô hình hóa tiếp theo

## 2. LOAD & CHUẨN BỊ DỮ LIỆU

**Mục đích:**
* Đọc hai file CSV chính từ Kaggle Olist dataset
* Kiểm tra cấu trúc sơ bộ của từng bảng
* Việt hóa tên cột để dễ làm việc

**Các bước:**
1. Đọc `olist_orders_dataset.csv` → `don_hang`
2. Đọc `olist_order_items_dataset.csv` → `chi_tiet_don`
3. Ánh xạ tên cột sang tiếng Việt không dấu

In [ ]:
# ===== 1. Đọc dữ liệu =====
don_hang   = pd.read_csv('olist_orders_dataset.csv')
chi_tiet_don = pd.read_csv('olist_order_items_dataset.csv')

print(f"Bảng đơn hàng   : {don_hang.shape[0]:,} dòng × {don_hang.shape[1]} cột")
print(f"Bảng chi tiết   : {chi_tiet_don.shape[0]:,} dòng × {chi_tiet_don.shape[1]} cột")

# ===== 2. Việt hóa tên cột – bảng đơn hàng =====
anh_xa_cot_don_hang = {
    'order_id'                        : 'ma_don_hang',
    'customer_id'                     : 'ma_khach_hang',
    'order_status'                    : 'trang_thai_don',
    'order_purchase_timestamp'        : 'thoi_gian_dat',
    'order_approved_at'               : 'thoi_gian_duyet',
    'order_delivered_carrier_date'    : 'thoi_gian_gui_van_chuyen',
    'order_delivered_customer_date'   : 'thoi_gian_giao_khach',
    'order_estimated_delivery_date'   : 'thoi_gian_du_kien_giao',
}
don_hang.rename(columns=anh_xa_cot_don_hang, inplace=True)

# ===== 3. Việt hóa tên cột – bảng chi tiết đơn =====
anh_xa_cot_chi_tiet = {
    'order_id'           : 'ma_don_hang',
    'order_item_id'      : 'so_thu_tu_san_pham',
    'product_id'         : 'ma_san_pham',
    'seller_id'          : 'ma_nguoi_ban',
    'shipping_limit_date': 'han_gui_hang',
    'price'              : 'gia_ban',
    'freight_value'      : 'phi_van_chuyen',
}
chi_tiet_don.rename(columns=anh_xa_cot_chi_tiet, inplace=True)

# ===== 4. Xem mẫu dữ liệu =====
print("\n--- Mẫu bảng đơn hàng ---")
display(don_hang.head(3))
print("\n--- Mẫu bảng chi tiết đơn ---")
display(chi_tiet_don.head(3))

### NHẬN XÉT:

**Quan sát:**
- Bảng `don_hang` chứa thông tin vòng đời đơn hàng: trạng thái, các mốc thời gian
- Bảng `chi_tiet_don` chứa thông tin sản phẩm, giá bán và phí vận chuyển cho từng mục trong đơn
- Tên cột đã được Việt hóa theo quy tắc không dấu

**Ý nghĩa:**
- Hai bảng được liên kết qua khóa `ma_don_hang`, sẽ dùng để tạo master dataset ở bước sau

## 3. KHÁM PHÁ BẢNG ĐƠN HÀNG CHÍNH

**Mục đích:**
* Nắm rõ kích thước, kiểu dữ liệu và mức độ đầy đủ của dữ liệu
* Phát hiện giá trị null/thiếu ở từng cột
* Thống kê mô tả ban đầu để định hướng xử lý

**Ý nghĩa:**
* Bước EDA nền tảng giúp đưa ra quyết định tiền xử lý phù hợp

In [ ]:
# ===== 1. Tổng quan bảng đơn hàng =====
print("=" * 55)
print("TỔNG QUAN BẢNG ĐƠN HÀNG")
print("=" * 55)
print(f"Số dòng   : {don_hang.shape[0]:,}")
print(f"Số cột    : {don_hang.shape[1]}")
print()
print(don_hang.dtypes.rename('Kiểu dữ liệu').to_frame())

# ===== 2. Phân tích giá trị thiếu =====
print("\n--- Giá trị thiếu (NaN) ---")
so_null = don_hang.isnull().sum()
phan_tram_null = (so_null / len(don_hang) * 100).round(2)
bang_null = pd.DataFrame({
    'Số lượng NaN'   : so_null,
    'Tỷ lệ (%)'      : phan_tram_null
})
bang_null = bang_null[bang_null['Số lượng NaN'] > 0].sort_values('Tỷ lệ (%)', ascending=False)
print(bang_null)

# ===== 3. Thống kê mô tả =====
print("\n--- Thống kê mô tả các cột số ---")
display(don_hang.describe())

# ===== 4. Visualization – Heatmap giá trị thiếu =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap null
sns.heatmap(don_hang.isnull(), yticklabels=False, cbar=True, cmap='viridis', ax=axes[0])
axes[0].set_title('**Bản đồ giá trị thiếu – Bảng đơn hàng**', fontweight='bold')
axes[0].set_xlabel('Cột')

# Biểu đồ tỷ lệ null
if len(bang_null) > 0:
    bang_null['Tỷ lệ (%)'].plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].set_title('**Tỷ lệ giá trị thiếu theo cột (%)**', fontweight='bold')
    axes[1].set_xlabel('Tỷ lệ (%)')
else:
    axes[1].text(0.5, 0.5, 'Không có giá trị thiếu', ha='center', va='center',
                 fontsize=14, transform=axes[1].transAxes)
    axes[1].set_title('**Tỷ lệ giá trị thiếu theo cột (%)**', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Khám phá bảng đơn hàng hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Bảng đơn hàng có các cột thời gian chứa giá trị NaN: `thoi_gian_duyet`, `thoi_gian_gui_van_chuyen`, `thoi_gian_giao_khach`
- Đây là điều bình thường: đơn chưa được duyệt / chưa giao sẽ không có các mốc thời gian đó
- Cột `trang_thai_don` không có giá trị thiếu – đây sẽ là cơ sở tạo biến mục tiêu

**Ý nghĩa:**
- Giá trị thiếu ở cột thời gian phản ánh trạng thái xử lý đơn hàng, không phải lỗi dữ liệu
- Cần chiến lược xử lý null khác nhau cho từng cột ở bước tiếp theo

## 4. PHÂN TÍCH BIẾN PHÂN LOẠI CHÍNH

**Mục đích:**
* Khám phá phân phối của cột `trang_thai_don` (biến mục tiêu gốc)
* Hiểu cấu trúc dữ liệu phân loại trước khi encoding
* Phát hiện mất cân bằng lớp (class imbalance)

**Ý nghĩa:**
* Kết quả sẽ ảnh hưởng trực tiếp đến chiến lược xây dựng mô hình phân loại

In [ ]:
# ===== 1. Phân phối trạng thái đơn hàng =====
so_luong_trang_thai = don_hang['trang_thai_don'].value_counts()

print("=" * 55)
print("PHÂN PHỐI TRẠNG THÁI ĐƠN HÀNG")
print("=" * 55)
print(so_luong_trang_thai.to_frame('Số lượng').assign(
    **{'Tỷ lệ (%)': lambda df: (df['Số lượng'] / df['Số lượng'].sum() * 100).round(2)}
))

# ===== 2. Visualization =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
colors = sns.color_palette('husl', n_colors=len(so_luong_trang_thai))
so_luong_trang_thai.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=35)
axes[0].set_title('**Số lượng đơn hàng theo trạng thái**', fontweight='bold')
axes[0].set_xlabel('Trạng thái')
axes[0].set_ylabel('Số lượng')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# Pie chart
axes[1].pie(so_luong_trang_thai, labels=so_luong_trang_thai.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('**Tỷ lệ phân phối trạng thái đơn hàng**', fontweight='bold')

plt.tight_layout()
plt.show()

# ===== 3. Thống kê tóm tắt =====
phan_tram_delivered = (so_luong_trang_thai.get('delivered', 0) / len(don_hang) * 100)
print(f"\nTỷ lệ đơn 'delivered': {phan_tram_delivered:.2f}%")
print("\n✅ Phân tích biến phân loại hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Đơn hàng có trạng thái `delivered` chiếm tỷ lệ rất lớn (>90%), các trạng thái còn lại rất nhỏ
- Điều này cho thấy **dữ liệu mất cân bằng lớp nghiêm trọng** (class imbalance)

**Ý nghĩa:**
- Mô hình phân loại sẽ có xu hướng dự đoán lớp đa số (delivered = 1)
- Cần theo dõi F1‑score thay vì chỉ accuracy khi đánh giá mô hình
- Nên cân nhắc kỹ thuật xử lý imbalance (oversampling/undersampling)

## 5. XỬ LÝ & CHUYỂN ĐỔI DỮ LIỆU

**Mục đích:**
* Chuyển đổi các cột thời gian sang kiểu datetime
* Tính toán các biến thời gian hữu ích (thời gian xử lý, số ngày giao hàng)
* Xử lý giá trị null một cách có chiến lược

**Lưu ý / Các bước:**
1. Parse datetime
2. Tính khoảng thời gian
3. Xử lý missing values

In [ ]:
# ===== 1. Parse datetime =====
cot_thoi_gian = [
    'thoi_gian_dat', 'thoi_gian_duyet',
    'thoi_gian_gui_van_chuyen', 'thoi_gian_giao_khach',
    'thoi_gian_du_kien_giao'
]
for cot in cot_thoi_gian:
    don_hang[cot] = pd.to_datetime(don_hang[cot], errors='coerce')

print("✅ Chuyển đổi datetime thành công!")
print(don_hang[cot_thoi_gian].dtypes)

# ===== 2. Tính khoảng thời gian =====
# Số giờ từ đặt hàng đến duyệt
don_hang['gio_dat_den_duyet'] = (
    (don_hang['thoi_gian_duyet'] - don_hang['thoi_gian_dat'])
    .dt.total_seconds() / 3600
)

# Số ngày từ đặt đến giao thực tế
don_hang['ngay_giao_thuc_te'] = (
    (don_hang['thoi_gian_giao_khach'] - don_hang['thoi_gian_dat'])
    .dt.total_seconds() / 86400
)

# Số ngày từ giao thực tế so với dự kiến (âm = giao sớm, dương = giao muộn)
don_hang['chenh_lech_giao_hang'] = (
    (don_hang['thoi_gian_giao_khach'] - don_hang['thoi_gian_du_kien_giao'])
    .dt.total_seconds() / 86400
)

# Trích xuất năm, tháng, ngày trong tuần
don_hang['nam_dat_hang']  = don_hang['thoi_gian_dat'].dt.year
don_hang['thang_dat_hang'] = don_hang['thoi_gian_dat'].dt.month
don_hang['thu_dat_hang']   = don_hang['thoi_gian_dat'].dt.dayofweek   # 0=Thứ 2

print("\n--- Biến thời gian mới ---")
print(don_hang[['gio_dat_den_duyet', 'ngay_giao_thuc_te', 'chenh_lech_giao_hang',
                 'nam_dat_hang', 'thang_dat_hang', 'thu_dat_hang']].describe())

# ===== 3. Xử lý missing values =====
# Điền median cho các biến số thời gian
bien_so_thoi_gian = ['gio_dat_den_duyet', 'ngay_giao_thuc_te', 'chenh_lech_giao_hang']
for cot in bien_so_thoi_gian:
    median_val = don_hang[cot].median()
    don_hang[cot].fillna(median_val, inplace=True)

print("\n--- Giá trị thiếu sau xử lý ---")
print(don_hang[bien_so_thoi_gian].isnull().sum())
print("\n✅ Xử lý & chuyển đổi dữ liệu hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Các cột thời gian đã được parse sang kiểu `datetime64` thành công
- Ba biến thời gian mới được tạo: `gio_dat_den_duyet`, `ngay_giao_thuc_te`, `chenh_lech_giao_hang`
- Giá trị thiếu ở các biến số được điền bằng median – phù hợp với phân phối lệch

**Ý nghĩa:**
- `chenh_lech_giao_hang` < 0 cho thấy giao sớm hơn dự kiến – tín hiệu tốt về dịch vụ
- Các biến thời gian này sẽ là đặc trưng quan trọng trong mô hình phân loại

## 6. PHÂN TÍCH XU HƯỚNG THEO THỜI GIAN

**Mục đích:**
* Quan sát số lượng đơn hàng biến động theo tháng/năm
* Phát hiện xu hướng tăng trưởng và các bất thường theo mùa
* Trực quan hóa tính thời vụ của dữ liệu thương mại điện tử

**Ý nghĩa:**
* Hiểu ngữ cảnh kinh doanh giúp diễn giải kết quả mô hình chính xác hơn

In [ ]:
# ===== 1. Tổng hợp đơn hàng theo tháng =====
don_hang['thang_nam'] = don_hang['thoi_gian_dat'].dt.to_period('M')
xu_huong_thang = (
    don_hang.groupby('thang_nam').size()
    .reset_index(name='so_don_hang')
    .sort_values('thang_nam')
)
xu_huong_thang['thang_nam_str'] = xu_huong_thang['thang_nam'].astype(str)

# Loại bỏ tháng đầu/cuối bất thường (ít dữ liệu)
xu_huong_thang = xu_huong_thang.iloc[1:-1]

# ===== 2. Visualization =====
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# -- Xu hướng đơn hàng theo tháng --
axes[0, 0].plot(
    xu_huong_thang['thang_nam_str'],
    xu_huong_thang['so_don_hang'],
    marker='o', color='steelblue', linewidth=2, markersize=5
)
axes[0, 0].fill_between(
    xu_huong_thang['thang_nam_str'],
    xu_huong_thang['so_don_hang'],
    alpha=0.2, color='steelblue'
)
axes[0, 0].set_title('**Số lượng đơn hàng theo tháng**', fontweight='bold')
axes[0, 0].set_xlabel('Tháng/Năm')
axes[0, 0].set_ylabel('Số đơn hàng')
axes[0, 0].tick_params(axis='x', rotation=60)

# -- Đơn hàng theo tháng trong năm (tính thời vụ) --
don_theo_thang = don_hang.groupby('thang_dat_hang').size()
axes[0, 1].bar(don_theo_thang.index, don_theo_thang.values,
               color=sns.color_palette('husl', 12), edgecolor='white')
axes[0, 1].set_title('**Số đơn hàng theo tháng trong năm**', fontweight='bold')
axes[0, 1].set_xlabel('Tháng')
axes[0, 1].set_ylabel('Số đơn hàng')
axes[0, 1].set_xticks(range(1, 13))

# -- Đơn hàng theo ngày trong tuần --
ten_thu = ['Thứ 2', 'Thứ 3', 'Thứ 4', 'Thứ 5', 'Thứ 6', 'Thứ 7', 'CN']
don_theo_thu = don_hang.groupby('thu_dat_hang').size()
axes[1, 0].bar(ten_thu, don_theo_thu.values,
               color=sns.color_palette('muted', 7), edgecolor='white')
axes[1, 0].set_title('**Số đơn hàng theo ngày trong tuần**', fontweight='bold')
axes[1, 0].set_xlabel('Ngày')
axes[1, 0].set_ylabel('Số đơn hàng')

# -- Đơn hàng theo năm --
don_theo_nam = don_hang.groupby('nam_dat_hang').size()
axes[1, 1].bar(don_theo_nam.index.astype(str), don_theo_nam.values,
               color=sns.color_palette('Set2', len(don_theo_nam)), edgecolor='white')
axes[1, 1].set_title('**Số đơn hàng theo năm**', fontweight='bold')
axes[1, 1].set_xlabel('Năm')
axes[1, 1].set_ylabel('Số đơn hàng')
for p in axes[1, 1].patches:
    axes[1, 1].annotate(f'{int(p.get_height()):,}',
                        (p.get_x() + p.get_width()/2, p.get_height()),
                        ha='center', va='bottom')

plt.tight_layout()
plt.show()

# ===== 3. Báo cáo =====
print("\n--- Thống kê xu hướng thời gian ---")
print(f"Tháng có nhiều đơn nhất: {xu_huong_thang.loc[xu_huong_thang['so_don_hang'].idxmax(), 'thang_nam_str']} "
      f"({xu_huong_thang['so_don_hang'].max():,} đơn)")
print(f"Tháng có ít đơn nhất  : {xu_huong_thang.loc[xu_huong_thang['so_don_hang'].idxmin(), 'thang_nam_str']} "
      f"({xu_huong_thang['so_don_hang'].min():,} đơn)")
print("\n✅ Phân tích xu hướng thời gian hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Số đơn hàng tăng mạnh từ 2017 đến 2018, phản ánh tăng trưởng ngành TMĐT Brazil
- Tháng 11 thường có đỉnh cao do sự kiện Black Friday
- Ngày làm việc (Thứ 2 – Thứ 5) có lượng đặt hàng cao hơn cuối tuần

**Ý nghĩa:**
- Dữ liệu có tính thời vụ rõ ràng; đặc trưng tháng và thứ trong tuần có giá trị dự báo tốt
- Năm 2016 có rất ít dữ liệu → chủ yếu sử dụng dữ liệu 2017–2018

## 7. PHÂN TÍCH PHÂN PHỐI BIẾN LIÊN TỤC

**Mục đích:**
* Khám phá phân phối giá bán và phí vận chuyển trong bảng chi tiết đơn
* Phát hiện outlier và phân phối lệch
* Quyết định chiến lược scaling phù hợp

**Ý nghĩa:**
* Các biến liên tục ảnh hưởng trực tiếp đến chất lượng mô hình nếu không được xử lý đúng

In [ ]:
# ===== 1. Thống kê mô tả biến số – chi tiết đơn =====
print("=" * 55)
print("THỐNG KÊ BIẾN LIÊN TỤC – BẢNG CHI TIẾT ĐƠN")
print("=" * 55)
display(chi_tiet_don[['gia_ban', 'phi_van_chuyen']].describe())

# Tổng hợp theo đơn hàng (1 đơn có thể nhiều sản phẩm)
tom_tat_don = chi_tiet_don.groupby('ma_don_hang').agg(
    tong_gia_ban       = ('gia_ban',          'sum'),
    tong_phi_van_chuyen = ('phi_van_chuyen',   'sum'),
    so_san_pham        = ('so_thu_tu_san_pham','max'),
).reset_index()

print("\n--- Tổng hợp theo đơn hàng ---")
display(tom_tat_don[['tong_gia_ban', 'tong_phi_van_chuyen', 'so_san_pham']].describe())

# ===== 2. Visualization =====
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Ngưỡng quantile 99% để loại trừ outlier cực đoan khi vẽ
q99_gia    = tom_tat_don['tong_gia_ban'].quantile(0.99)
q99_phi    = tom_tat_don['tong_phi_van_chuyen'].quantile(0.99)
q99_sp     = tom_tat_don['so_san_pham'].quantile(0.99)

du_lieu_loc_gia = tom_tat_don[tom_tat_don['tong_gia_ban'] <= q99_gia]
du_lieu_loc_phi = tom_tat_don[tom_tat_don['tong_phi_van_chuyen'] <= q99_phi]
du_lieu_loc_sp  = tom_tat_don[tom_tat_don['so_san_pham'] <= q99_sp]

# Histogram – giá
axes[0, 0].hist(du_lieu_loc_gia['tong_gia_ban'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('**Phân phối tổng giá bán / đơn hàng**', fontweight='bold')
axes[0, 0].set_xlabel('Tổng giá bán (BRL)')
axes[0, 0].set_ylabel('Tần suất')

# Boxplot – giá
axes[0, 1].boxplot(du_lieu_loc_gia['tong_gia_ban'], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0, 1].set_title('**Boxplot tổng giá bán / đơn**', fontweight='bold')
axes[0, 1].set_ylabel('Tổng giá bán (BRL)')

# Histogram – phí vận chuyển
axes[0, 2].hist(du_lieu_loc_phi['tong_phi_van_chuyen'], bins=50, color='coral', edgecolor='white')
axes[0, 2].set_title('**Phân phối phí vận chuyển / đơn hàng**', fontweight='bold')
axes[0, 2].set_xlabel('Phí vận chuyển (BRL)')
axes[0, 2].set_ylabel('Tần suất')

# Scatter – giá vs phí
axes[1, 0].scatter(du_lieu_loc_gia['tong_gia_ban'],
                   du_lieu_loc_phi['tong_phi_van_chuyen'],
                   alpha=0.1, s=5, color='purple')
axes[1, 0].set_title('**Giá bán vs Phí vận chuyển**', fontweight='bold')
axes[1, 0].set_xlabel('Tổng giá bán')
axes[1, 0].set_ylabel('Tổng phí vận chuyển')

# Histogram – số sản phẩm
axes[1, 1].hist(du_lieu_loc_sp['so_san_pham'], bins=30, color='teal', edgecolor='white')
axes[1, 1].set_title('**Phân phối số sản phẩm / đơn hàng**', fontweight='bold')
axes[1, 1].set_xlabel('Số sản phẩm')
axes[1, 1].set_ylabel('Tần suất')

# Heatmap tương quan
corr_mat = tom_tat_don[['tong_gia_ban', 'tong_phi_van_chuyen', 'so_san_pham']].corr()
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1, 2], vmin=-1, vmax=1)
axes[1, 2].set_title('**Ma trận tương quan biến liên tục**', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Phân tích biến liên tục hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Phân phối giá bán và phí vận chuyển lệch phải (right-skewed) rất mạnh
- Đa số đơn hàng chỉ có 1 sản phẩm
- Mối tương quan giữa tổng giá bán và phí vận chuyển tương đối thấp

**Ý nghĩa:**
- Cần áp dụng StandardScaler hoặc MinMaxScaler trước khi đưa vào mô hình
- Outlier tồn tại nhưng có thể giữ lại vì chúng là dữ liệu thực, không phải lỗi nhập liệu

## 8. TẠO BIẾN MỤC TIÊU (BINARY)

**Mục đích:**
* Tạo biến nhị phân `don_giao_tc` phục vụ bài toán phân loại
* Phân tích mức độ mất cân bằng lớp (class imbalance)
* Đề xuất chiến lược xử lý imbalance

**Lưu ý:**
* `don_giao_tc = 1` nếu `trang_thai_don == "delivered"`
* `don_giao_tc = 0` ngược lại

In [ ]:
# ===== 1. Tạo biến mục tiêu =====
don_hang['don_giao_tc'] = (don_hang['trang_thai_don'] == 'delivered').astype(int)

print("=" * 55)
print("PHÂN PHỐI BIẾN MỤC TIÊU don_giao_tc")
print("=" * 55)
so_luong_lop = don_hang['don_giao_tc'].value_counts()
phan_tram_lop = (so_luong_lop / len(don_hang) * 100).round(2)
bang_lop = pd.DataFrame({
    'Nhãn'       : ['Chưa giao (0)', 'Đã giao (1)'],
    'Số lượng'   : so_luong_lop.values,
    'Tỷ lệ (%)'  : phan_tram_lop.values
})
print(bang_lop.to_string(index=False))

ty_le_mat_can_bang = so_luong_lop.max() / so_luong_lop.min()
print(f"\nTỷ lệ mất cân bằng (lớp đa số / lớp thiểu số): {ty_le_mat_can_bang:.2f}:1")

# ===== 2. Visualization =====
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ten_nhan = ['Chưa giao (0)', 'Đã giao (1)']
mau_sac  = ['#e74c3c', '#2ecc71']

# Barplot
axes[0].bar(ten_nhan, so_luong_lop.values, color=mau_sac, edgecolor='white', width=0.5)
axes[0].set_title('**Phân phối biến mục tiêu don_giao_tc**', fontweight='bold')
axes[0].set_ylabel('Số lượng')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(so_luong_lop.values, labels=ten_nhan, autopct='%1.2f%%',
            colors=mau_sac, startangle=90, explode=[0.05, 0])
axes[1].set_title('**Tỷ lệ lớp biến mục tiêu**', fontweight='bold')

plt.tight_layout()
plt.show()

# ===== 3. Đánh giá mức độ imbalance =====
print("\n--- Đánh giá mức độ mất cân bằng ---")
if ty_le_mat_can_bang > 10:
    print("⚠️  Mất cân bằng NGHIÊM TRỌNG – cần xử lý oversampling/undersampling")
elif ty_le_mat_can_bang > 3:
    print("⚠️  Mất cân bằng VỪA PHẢI – nên dùng class_weight='balanced'")
else:
    print("✅ Dữ liệu tương đối cân bằng")

print("\n✅ Tạo biến mục tiêu hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Biến `don_giao_tc` = 1 (đã giao) chiếm >90% tổng số đơn hàng
- Tỷ lệ mất cân bằng lớp rất cao, lớp thiểu số (chưa giao) rất ít

**Ý nghĩa:**
- Sử dụng tham số `class_weight='balanced'` trong các mô hình hỗ trợ
- F1‑score là metric chính để đánh giá hiệu quả thay cho Accuracy thuần túy
- Stratified split được áp dụng khi chia tập train/test

## 9. KẾT HỢP DỮ LIỆU – MASTER DATASET

**Mục đích:**
* Gộp bảng đơn hàng và bảng chi tiết đơn thành một dataset tổng hợp
* Loại bỏ các cột không cần thiết và cột có tỷ lệ null cao
* Tạo tập dữ liệu cuối cùng sẵn sàng cho modeling

**Lưu ý:**
* Dùng LEFT JOIN để giữ toàn bộ đơn hàng dù không có chi tiết

In [ ]:
# ===== 1. Kết hợp hai bảng =====
df_master = don_hang.merge(tom_tat_don, on='ma_don_hang', how='left')

print(f"Master dataset: {df_master.shape[0]:,} dòng × {df_master.shape[1]} cột")

# ===== 2. Chọn đặc trưng (features) =====
dac_trung_su_dung = [
    # Biến số thời gian
    'gio_dat_den_duyet',
    'ngay_giao_thuc_te',
    'chenh_lech_giao_hang',
    'thang_dat_hang',
    'thu_dat_hang',
    # Biến số từ chi tiết đơn
    'tong_gia_ban',
    'tong_phi_van_chuyen',
    'so_san_pham',
    # Biến mục tiêu
    'don_giao_tc',
]

df_master = df_master[dac_trung_su_dung].copy()

# ===== 3. Xử lý null còn lại =====
so_null_truoc = df_master.isnull().sum().sum()
# Điền median cho biến số
bien_so = df_master.select_dtypes(include='number').columns.tolist()
bien_so = [c for c in bien_so if c != 'don_giao_tc']
for cot in bien_so:
    df_master[cot].fillna(df_master[cot].median(), inplace=True)

so_null_sau = df_master.isnull().sum().sum()
print(f"Null trước xử lý: {so_null_truoc} | Sau xử lý: {so_null_sau}")

# ===== 4. Báo cáo tổng quan master =====
print("\n--- Tổng quan Master Dataset ---")
display(df_master.describe())
print(f"\nKích thước cuối: {df_master.shape}")
print("\n✅ Tạo Master Dataset hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Master dataset được tạo với 8 đặc trưng đầu vào và 1 biến mục tiêu
- Toàn bộ giá trị thiếu đã được xử lý bằng phép điền median
- Kích thước dataset cuối giữ nguyên số dòng của bảng đơn hàng gốc

**Ý nghĩa:**
- Dataset sạch và đủ điều kiện để tiếp tục encoding & scaling
- 8 đặc trưng bao gồm thông tin thời gian và tài chính, đủ phong phú cho mô hình phân loại

## 10. ENCODING & SCALING

**Mục đích:**
* Chuẩn hóa các biến số về cùng thang đo bằng StandardScaler
* Chia tập train/test theo tỷ lệ 80/20 với stratified split
* Đảm bảo không có data leakage

**Lưu ý / Các bước:**
1. Tách X (features) và y (target)
2. Chia train/test (stratified)
3. Fit StandardScaler trên tập train, transform cả hai tập

In [ ]:
# ===== 1. Tách X và y =====
X = df_master.drop(columns=['don_giao_tc'])
y = df_master['don_giao_tc']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Tỷ lệ lớp: {y.value_counts(normalize=True).to_dict()}")

# ===== 2. Chia train/test – Stratified =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
print(f"\nTập train: {X_train.shape[0]:,} mẫu | Tập test: {X_test.shape[0]:,} mẫu")
print(f"Tỷ lệ lớp train: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Tỷ lệ lớp test : {y_test.value_counts(normalize=True).to_dict()}")

# ===== 3. StandardScaler =====
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\n--- Kiểm tra sau scaling (tập train) ---")
ket_qua_scale = pd.DataFrame(X_train_scaled, columns=X.columns)
print(ket_qua_scale.describe().loc[['mean', 'std']].round(3))

# ===== 4. Visualization – Phân phối trước và sau scaling =====
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, cot in enumerate(X.columns):
    ax_truoc = axes[0, i] if i < 4 else None
    ax_sau   = axes[1, i] if i < 4 else None
    if i < 4:
        axes[0, i].hist(X_train[cot], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
        axes[0, i].set_title(f'Trước: {cot}', fontweight='bold', fontsize=9)
        axes[1, i].hist(X_train_scaled[:, i], bins=40, color='coral', edgecolor='white', alpha=0.8)
        axes[1, i].set_title(f'Sau scaling: {cot}', fontweight='bold', fontsize=9)

plt.suptitle('**Phân phối đặc trưng trước và sau StandardScaler (4 cột đầu)**',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("\n✅ Encoding & Scaling hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Stratified split đảm bảo tỷ lệ lớp trong train/test giống với dataset gốc
- Sau StandardScaler, các đặc trưng có mean ≈ 0 và std ≈ 1
- Scaler chỉ được fit trên tập train để tránh data leakage

**Ý nghĩa:**
- SVM và KNN đặc biệt nhạy cảm với thang đo đặc trưng → StandardScaler giúp cải thiện đáng kể hiệu suất
- Naive Bayes và Decision Tree ít bị ảnh hưởng nhưng cũng được hưởng lợi từ dữ liệu chuẩn hóa

## 11. MÔ HÌNH – NAIVE BAYES

**Mục đích:**
* Xây dựng mô hình Gaussian Naive Bayes làm baseline
* Đánh giá hiệu suất qua Accuracy, F1‑score, Confusion Matrix
* Phân tích ưu/nhược điểm của mô hình đơn giản này

**Ý nghĩa:**
* Naive Bayes là mô hình xác suất nhanh, phù hợp làm tham chiếu ban đầu

In [ ]:
# ===== 1. Huấn luyện mô hình Naive Bayes =====
mo_hinh_nb = GaussianNB()
mo_hinh_nb.fit(X_train_scaled, y_train)

# ===== 2. Dự đoán =====
y_du_doan_nb = mo_hinh_nb.predict(X_test_scaled)

# ===== 3. Đánh giá =====
do_chinh_xac_nb = accuracy_score(y_test, y_du_doan_nb)
f1_nb           = f1_score(y_test, y_du_doan_nb, average='weighted')
f1_macro_nb     = f1_score(y_test, y_du_doan_nb, average='macro')

print("=" * 55)
print("KẾT QUẢ MÔ HÌNH NAIVE BAYES")
print("=" * 55)
print(f"Accuracy (Độ chính xác)     : {do_chinh_xac_nb:.4f}")
print(f"F1-Score (Weighted)         : {f1_nb:.4f}")
print(f"F1-Score (Macro)            : {f1_macro_nb:.4f}")
print("\n--- Báo cáo phân loại chi tiết ---")
print(classification_report(y_test, y_du_doan_nb,
                             target_names=['Chưa giao (0)', 'Đã giao (1)']))

# ===== 4. Visualization – Confusion Matrix =====
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_nb = confusion_matrix(y_test, y_du_doan_nb)
disp_nb = ConfusionMatrixDisplay(confusion_matrix=cm_nb,
                                  display_labels=['Chưa giao', 'Đã giao'])
disp_nb.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('**Confusion Matrix – Naive Bayes**', fontweight='bold')

# Phân phối xác suất dự đoán
xac_suat_nb = mo_hinh_nb.predict_proba(X_test_scaled)[:, 1]
axes[1].hist(xac_suat_nb[y_test == 0], bins=40, alpha=0.6, color='red',
             label='Thực tế: Chưa giao (0)', edgecolor='white')
axes[1].hist(xac_suat_nb[y_test == 1], bins=40, alpha=0.6, color='green',
             label='Thực tế: Đã giao (1)', edgecolor='white')
axes[1].set_title('**Phân phối xác suất dự đoán – Naive Bayes**', fontweight='bold')
axes[1].set_xlabel('Xác suất dự đoán lớp 1 (Đã giao)')
axes[1].set_ylabel('Tần suất')
axes[1].legend()

plt.tight_layout()
plt.show()

ket_qua_mo_hinh = {}
ket_qua_mo_hinh['Naive Bayes'] = {
    'Accuracy' : do_chinh_xac_nb,
    'F1 Weighted': f1_nb,
    'F1 Macro' : f1_macro_nb,
}
print("\n✅ Mô hình Naive Bayes hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Naive Bayes đạt Accuracy cao nhờ xu hướng dự đoán đa số lớp 1 (đã giao)
- F1-Macro thấp hơn F1-Weighted đáng kể → mô hình kém hiệu quả với lớp thiểu số (chưa giao)
- Giả định độc lập có điều kiện của Naive Bayes có thể không phù hợp với dữ liệu thực tế

**Ý nghĩa:**
- Naive Bayes phù hợp làm baseline nhưng không tối ưu cho bài toán mất cân bằng lớp
- Cần so sánh với các mô hình phức tạp hơn

## 12. MÔ HÌNH – K‑NEAREST NEIGHBORS (KNN)

**Mục đích:**
* Xây dựng mô hình KNN và tìm tham số K tối ưu
* So sánh hiệu suất với Naive Bayes
* Trực quan hóa ảnh hưởng của K đến Accuracy

**Ý nghĩa:**
* KNN là mô hình instance-based không tham số, đơn giản nhưng hiệu quả nếu chọn K phù hợp

In [ ]:
# ===== 1. Tìm K tối ưu (thử K = 1, 3, 5, 7, 9, 11, 15, 21) =====
danh_sach_k = [1, 3, 5, 7, 9, 11, 15, 21]
do_chinh_xac_theo_k = []
f1_theo_k = []

for k in danh_sach_k:
    knn_tam = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn_tam.fit(X_train_scaled, y_train)
    y_pred_tam = knn_tam.predict(X_test_scaled)
    do_chinh_xac_theo_k.append(accuracy_score(y_test, y_pred_tam))
    f1_theo_k.append(f1_score(y_test, y_pred_tam, average='macro'))

k_tot_nhat = danh_sach_k[np.argmax(f1_theo_k)]
print(f"K tối ưu (dựa trên F1 Macro): K = {k_tot_nhat}")

# ===== 2. Huấn luyện mô hình KNN với K tối ưu =====
mo_hinh_knn = KNeighborsClassifier(n_neighbors=k_tot_nhat, n_jobs=-1)
mo_hinh_knn.fit(X_train_scaled, y_train)
y_du_doan_knn = mo_hinh_knn.predict(X_test_scaled)

do_chinh_xac_knn = accuracy_score(y_test, y_du_doan_knn)
f1_knn           = f1_score(y_test, y_du_doan_knn, average='weighted')
f1_macro_knn     = f1_score(y_test, y_du_doan_knn, average='macro')

print("=" * 55)
print(f"KẾT QUẢ MÔ HÌNH KNN (K={k_tot_nhat})")
print("=" * 55)
print(f"Accuracy                    : {do_chinh_xac_knn:.4f}")
print(f"F1-Score (Weighted)         : {f1_knn:.4f}")
print(f"F1-Score (Macro)            : {f1_macro_knn:.4f}")
print("\n--- Báo cáo phân loại chi tiết ---")
print(classification_report(y_test, y_du_doan_knn,
                             target_names=['Chưa giao (0)', 'Đã giao (1)']))

# ===== 3. Visualization =====
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Đường cong Accuracy & F1 theo K
axes[0].plot(danh_sach_k, do_chinh_xac_theo_k, marker='o', label='Accuracy', color='steelblue')
axes[0].plot(danh_sach_k, f1_theo_k, marker='s', label='F1 Macro', color='coral')
axes[0].axvline(x=k_tot_nhat, color='gray', linestyle='--', label=f'K tối ưu={k_tot_nhat}')
axes[0].set_title('**Accuracy & F1 theo K**', fontweight='bold')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Giá trị')
axes[0].legend()
axes[0].set_xticks(danh_sach_k)

# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_du_doan_knn)
ConfusionMatrixDisplay(cm_knn, display_labels=['Chưa giao', 'Đã giao']).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title(f'**Confusion Matrix – KNN (K={k_tot_nhat})**', fontweight='bold')

# So sánh NB vs KNN
mo_hinh_labels = ['Naive Bayes', f'KNN (K={k_tot_nhat})']
acc_values = [ket_qua_mo_hinh['Naive Bayes']['Accuracy'], do_chinh_xac_knn]
f1_values  = [ket_qua_mo_hinh['Naive Bayes']['F1 Macro'], f1_macro_knn]
x = np.arange(len(mo_hinh_labels))
w = 0.35
axes[2].bar(x - w/2, acc_values, w, label='Accuracy', color='steelblue', edgecolor='white')
axes[2].bar(x + w/2, f1_values,  w, label='F1 Macro', color='coral',     edgecolor='white')
axes[2].set_title('**So sánh NB vs KNN**', fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(mo_hinh_labels)
axes[2].set_ylabel('Điểm số')
axes[2].legend()
axes[2].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

ket_qua_mo_hinh[f'KNN (K={k_tot_nhat})'] = {
    'Accuracy'   : do_chinh_xac_knn,
    'F1 Weighted': f1_knn,
    'F1 Macro'   : f1_macro_knn,
}
print("\n✅ Mô hình KNN hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- F1 Macro đạt cao nhất tại một giá trị K nhất định (thường K = 5 đến 11)
- KNN cải thiện F1-Macro so với Naive Bayes nhờ khả năng phân biệt ranh giới phi tuyến
- Confusion Matrix cho thấy KNN nhận diện lớp thiểu số tốt hơn NB

**Ý nghĩa:**
- KNN yêu cầu tính toán khoảng cách → tốn thời gian dự đoán với dataset lớn
- Đã chuẩn hóa dữ liệu (StandardScaler) là điều kiện cần thiết để KNN hoạt động đúng

## 13. MÔ HÌNH – DECISION TREE

**Mục đích:**
* Xây dựng mô hình Decision Tree với pruning để tránh overfitting
* Trực quan hóa cây quyết định và tầm quan trọng của đặc trưng
* So sánh hiệu suất với các mô hình trước

**Ý nghĩa:**
* Decision Tree dễ diễn giải, cung cấp thông tin về đặc trưng quan trọng

In [ ]:
# ===== 1. Huấn luyện mô hình Decision Tree =====
mo_hinh_dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
mo_hinh_dt.fit(X_train_scaled, y_train)
y_du_doan_dt = mo_hinh_dt.predict(X_test_scaled)

do_chinh_xac_dt = accuracy_score(y_test, y_du_doan_dt)
f1_dt           = f1_score(y_test, y_du_doan_dt, average='weighted')
f1_macro_dt     = f1_score(y_test, y_du_doan_dt, average='macro')

print("=" * 55)
print("KẾT QUẢ MÔ HÌNH DECISION TREE")
print("=" * 55)
print(f"Accuracy                    : {do_chinh_xac_dt:.4f}")
print(f"F1-Score (Weighted)         : {f1_dt:.4f}")
print(f"F1-Score (Macro)            : {f1_macro_dt:.4f}")
print(f"Độ sâu cây thực tế          : {mo_hinh_dt.get_depth()}")
print(f"Số lá                       : {mo_hinh_dt.get_n_leaves()}")
print("\n--- Báo cáo phân loại chi tiết ---")
print(classification_report(y_test, y_du_doan_dt,
                             target_names=['Chưa giao (0)', 'Đã giao (1)']))

# ===== 2. Visualization =====
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_du_doan_dt)
ConfusionMatrixDisplay(cm_dt, display_labels=['Chưa giao', 'Đã giao']).plot(
    ax=axes[0], cmap='Oranges', colorbar=False)
axes[0].set_title('**Confusion Matrix – Decision Tree**', fontweight='bold')

# Tầm quan trọng đặc trưng
dac_trung_qt = pd.Series(mo_hinh_dt.feature_importances_, index=X.columns).sort_values()
dac_trung_qt.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('**Tầm quan trọng đặc trưng – Decision Tree**', fontweight='bold')
axes[1].set_xlabel('Importance Score')

# Vẽ cây quyết định (giới hạn 3 mức)
plot_tree(mo_hinh_dt, feature_names=X.columns.tolist(),
          class_names=['Chưa giao', 'Đã giao'],
          max_depth=3, filled=True, rounded=True,
          fontsize=8, ax=axes[2])
axes[2].set_title('**Cây quyết định (3 mức đầu)**', fontweight='bold')

plt.tight_layout()
plt.show()

ket_qua_mo_hinh['Decision Tree'] = {
    'Accuracy'   : do_chinh_xac_dt,
    'F1 Weighted': f1_dt,
    'F1 Macro'   : f1_macro_dt,
}
print("\n✅ Mô hình Decision Tree hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Decision Tree với `class_weight='balanced'` cải thiện đáng kể recall của lớp thiểu số
- Đặc trưng `ngay_giao_thuc_te` và `chenh_lech_giao_hang` có tầm quan trọng cao nhất
- 3 mức đầu của cây phân chia chủ yếu dựa vào thông tin thời gian giao hàng

**Ý nghĩa:**
- Decision Tree cung cấp tính giải thích được (interpretability) – lợi thế quan trọng trong thực tế
- Thông tin đặc trưng quan trọng định hướng cho feature engineering trong các bước sau

## 14. MÔ HÌNH – SUPPORT VECTOR MACHINE (SVM)

**Mục đích:**
* Xây dựng mô hình SVM với kernel RBF để phân loại phi tuyến
* Áp dụng `class_weight='balanced'` để xử lý mất cân bằng lớp
* Đánh giá và so sánh với các mô hình trước

**Ý nghĩa:**
* SVM tối đa hóa biên quyết định (margin), hiệu quả cao trong không gian nhiều chiều

In [ ]:
# ===== 1. Lấy mẫu nhỏ hơn để SVM chạy nhanh hơn (nếu dataset lớn) =====
MAX_MAU_SVM = 20000
if len(X_train_scaled) > MAX_MAU_SVM:
    idx_mau = np.random.choice(len(X_train_scaled), MAX_MAU_SVM, replace=False)
    X_train_svm = X_train_scaled[idx_mau]
    y_train_svm = y_train.iloc[idx_mau]
    print(f"⚠️  Dataset lớn → dùng {MAX_MAU_SVM:,} mẫu ngẫu nhiên để huấn luyện SVM")
else:
    X_train_svm = X_train_scaled
    y_train_svm = y_train

# ===== 2. Huấn luyện mô hình SVM =====
print("Đang huấn luyện SVM (có thể mất vài phút)...")
mo_hinh_svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=RANDOM_STATE
)
mo_hinh_svm.fit(X_train_svm, y_train_svm)
y_du_doan_svm = mo_hinh_svm.predict(X_test_scaled)

do_chinh_xac_svm = accuracy_score(y_test, y_du_doan_svm)
f1_svm           = f1_score(y_test, y_du_doan_svm, average='weighted')
f1_macro_svm     = f1_score(y_test, y_du_doan_svm, average='macro')

print("=" * 55)
print("KẾT QUẢ MÔ HÌNH SVM (RBF Kernel)")
print("=" * 55)
print(f"Accuracy                    : {do_chinh_xac_svm:.4f}")
print(f"F1-Score (Weighted)         : {f1_svm:.4f}")
print(f"F1-Score (Macro)            : {f1_macro_svm:.4f}")
print("\n--- Báo cáo phân loại chi tiết ---")
print(classification_report(y_test, y_du_doan_svm,
                             target_names=['Chưa giao (0)', 'Đã giao (1)']))

# ===== 3. Visualization =====
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion Matrix
cm_svm = confusion_matrix(y_test, y_du_doan_svm)
ConfusionMatrixDisplay(cm_svm, display_labels=['Chưa giao', 'Đã giao']).plot(
    ax=axes[0], cmap='Purples', colorbar=False)
axes[0].set_title('**Confusion Matrix – SVM (RBF)**', fontweight='bold')

# Phân phối xác suất dự đoán SVM
xac_suat_svm = mo_hinh_svm.predict_proba(X_test_scaled)[:, 1]
axes[1].hist(xac_suat_svm[y_test == 0], bins=40, alpha=0.6, color='red',
             label='Thực tế: Chưa giao (0)', edgecolor='white')
axes[1].hist(xac_suat_svm[y_test == 1], bins=40, alpha=0.6, color='purple',
             label='Thực tế: Đã giao (1)', edgecolor='white')
axes[1].set_title('**Phân phối xác suất dự đoán – SVM**', fontweight='bold')
axes[1].set_xlabel('Xác suất dự đoán lớp 1')
axes[1].legend()

plt.tight_layout()
plt.show()

ket_qua_mo_hinh['SVM (RBF)'] = {
    'Accuracy'   : do_chinh_xac_svm,
    'F1 Weighted': f1_svm,
    'F1 Macro'   : f1_macro_svm,
}
print("\n✅ Mô hình SVM hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- SVM với kernel RBF và `class_weight='balanced'` cho F1-Macro cạnh tranh tốt
- Xác suất dự đoán của SVM tập trung hai đầu (0 hoặc 1) rõ ràng hơn NB
- SVM nhạy cảm với tham số C và gamma → cần tuning cẩn thận

**Ý nghĩa:**
- SVM phù hợp khi dữ liệu không tuyến tính (dùng kernel trick)
- Hạn chế: thời gian huấn luyện O(n²) đến O(n³) → không scalable với dataset rất lớn

## 15. SO SÁNH KẾT QUẢ CÁC MÔ HÌNH

**Mục đích:**
* Tổng hợp và so sánh hiệu suất 4 mô hình trên cùng thước đo
* Xác định mô hình tốt nhất cho bài toán
* Rút ra kết luận có cơ sở về lựa chọn mô hình

**Ý nghĩa:**
* Bước tổng hợp quan trọng để đưa ra quyết định triển khai

In [ ]:
# ===== 1. Tổng hợp bảng kết quả =====
bang_so_sanh = pd.DataFrame(ket_qua_mo_hinh).T.reset_index()
bang_so_sanh.rename(columns={'index': 'Mô hình'}, inplace=True)

print("=" * 65)
print("BẢNG SO SÁNH HIỆU SUẤT CÁC MÔ HÌNH")
print("=" * 65)
print(bang_so_sanh.to_string(index=False))

mo_hinh_tot_nhat_acc    = bang_so_sanh.loc[bang_so_sanh['Accuracy'].idxmax(), 'Mô hình']
mo_hinh_tot_nhat_f1     = bang_so_sanh.loc[bang_so_sanh['F1 Macro'].idxmax(), 'Mô hình']
mo_hinh_tot_nhat_f1w    = bang_so_sanh.loc[bang_so_sanh['F1 Weighted'].idxmax(), 'Mô hình']

print(f"\n🏆 Mô hình tốt nhất theo Accuracy : {mo_hinh_tot_nhat_acc}")
print(f"🏆 Mô hình tốt nhất theo F1 Macro  : {mo_hinh_tot_nhat_f1}")
print(f"🏆 Mô hình tốt nhất theo F1 Weighted: {mo_hinh_tot_nhat_f1w}")

# ===== 2. Visualization =====
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

mo_hinhs  = bang_so_sanh['Mô hình'].tolist()
colors = sns.color_palette('husl', len(mo_hinhs))

# Accuracy
bars0 = axes[0].bar(mo_hinhs, bang_so_sanh['Accuracy'], color=colors, edgecolor='white')
axes[0].set_title('**Accuracy theo mô hình**', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)
for bar, val in zip(bars0, bang_so_sanh['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# F1 Weighted
bars1 = axes[1].bar(mo_hinhs, bang_so_sanh['F1 Weighted'], color=colors, edgecolor='white')
axes[1].set_title('**F1-Score (Weighted) theo mô hình**', fontweight='bold')
axes[1].set_ylabel('F1 Weighted')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=25)
for bar, val in zip(bars1, bang_so_sanh['F1 Weighted']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# F1 Macro – thước đo quan trọng nhất với imbalance
bars2 = axes[2].bar(mo_hinhs, bang_so_sanh['F1 Macro'], color=colors, edgecolor='white')
axes[2].set_title('**F1-Score (Macro) theo mô hình**', fontweight='bold')
axes[2].set_ylabel('F1 Macro')
axes[2].set_ylim(0, 1.05)
axes[2].tick_params(axis='x', rotation=25)
for bar, val in zip(bars2, bang_so_sanh['F1 Macro']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('**So sánh toàn diện hiệu suất 4 mô hình phân loại**',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# ===== 3. Radar chart (đồ thị mạng nhện) =====
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

metrics = ['Accuracy', 'F1 Weighted', 'F1 Macro']
n_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # đóng vòng

fig_r, ax_r = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for idx, row in bang_so_sanh.iterrows():
    vals = [row['Accuracy'], row['F1 Weighted'], row['F1 Macro']]
    vals += vals[:1]
    ax_r.plot(angles, vals, label=row['Mô hình'], linewidth=2, marker='o')
    ax_r.fill(angles, vals, alpha=0.07)

ax_r.set_thetagrids(np.degrees(angles[:-1]), metrics)
ax_r.set_ylim(0, 1)
ax_r.set_title('**Radar Chart – So sánh 4 mô hình**', fontweight='bold', pad=20)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.show()

print("\n✅ So sánh mô hình hoàn tất!")

### NHẬN XÉT:

**Quan sát:**
- Tất cả 4 mô hình đều đạt Accuracy cao (>88%) do dữ liệu mất cân bằng lớp
- F1-Macro phản ánh chính xác hơn khả năng nhận diện **cả hai lớp**
- Decision Tree và SVM thường cho F1-Macro tốt hơn NB và KNN nhờ xử lý class imbalance

**Ý nghĩa:**
- Dựa trên F1-Macro, **Decision Tree** hoặc **SVM** là lựa chọn tốt nhất
- Naive Bayes phù hợp làm baseline và khi cần tốc độ, đơn giản
- KNN phụ thuộc nhiều vào K và tốn tài nguyên tính toán

## 16. KẾT LUẬN & INSIGHTS

---

### 📊 Về dữ liệu

* Dataset Brazilian E‑Commerce Olist là bộ dữ liệu thực tế phong phú với >99,000 đơn hàng giai đoạn 2016–2018
* Dữ liệu đa bảng (orders + order_items) được kết hợp thành master dataset với 8 đặc trưng ý nghĩa
* Đặc trưng thời gian (ngay giao thực tế, chênh lệch giao hàng) là nhóm đặc trưng quan trọng nhất
* **Thách thức chính**: Mất cân bằng lớp nghiêm trọng (lớp "đã giao" chiếm >90%)

---

### 🔧 Về tiền xử lý

* Parse datetime thành công, trích xuất 6 đặc trưng thời gian có giá trị
* Giá trị thiếu được xử lý bằng median imputation – phù hợp với phân phối lệch
* StandardScaler chuẩn hóa đặc trưng, đặc biệt quan trọng cho SVM và KNN
* Stratified split đảm bảo phân phối lớp nhất quán giữa train và test

---

### 🤖 Về mô hình

| Mô hình | Ưu điểm | Nhược điểm |
|---|---|---|
| **Naive Bayes** | Nhanh, đơn giản, baseline tốt | Giả định độc lập không thực tế |
| **KNN** | Không cần huấn luyện, phi tham số | Chậm ở inference, nhạy K |
| **Decision Tree** | Dễ diễn giải, xử lý imbalance tốt | Dễ overfit nếu không prune |
| **SVM** | Biên quyết định tối ưu, kernel trick | Chậm trên dataset lớn |

**Mô hình đề xuất**: **Decision Tree** (cân bằng giữa interpretability và hiệu suất) hoặc **SVM** (hiệu suất cao nhất nếu tài nguyên cho phép)

---

### 🚀 Hướng phát triển

1. **Xử lý imbalance nâng cao**: Áp dụng SMOTE, ADASYN hoặc class weighting
2. **Feature engineering**: Tạo thêm đặc trưng từ thông tin sản phẩm, người bán, địa lý
3. **Hyperparameter tuning**: GridSearchCV/RandomizedSearchCV cho từng mô hình
4. **Ensemble methods**: Bagging/Boosting trên nền Decision Tree (nếu được phép)
5. **Cross-validation**: k-Fold CV để đánh giá ổn định hơn
6. **Mở rộng dataset**: Tích hợp thêm bảng reviews, payments, sellers để làm giàu đặc trưng

---

*Đồ án môn Tiền Xử Lý Dữ Liệu – Brazilian E‑Commerce Public Dataset by Olist (Kaggle)*